# Motor Insurance Pricing & Risk Analytics

**Dataset:** freMTPL2 French Motor Third-Party Liability  
**Methods:** Portfolio analytics, Poisson frequency GLM, Gamma severity GLM, pure-premium estimation

This notebook is the reproducible analysis companion to the GitHub project. The raw data is intentionally not stored in the repository; run the downloader first or allow the notebook to download it.


## 1. Setup

The project separates technical pricing into:

\[
	ext{Pure Premium} = E[	ext{Claim Frequency}] 	imes E[	ext{Claim Severity}]
\]

Frequency is annualised using policy exposure. Severity is conditional on a positive claim.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from config import FREQ_PATH, SEV_PATH, PROCESSED_DIR, MODEL_DIR
from download_data import main as download_data
from analysis import load_data, portfolio_summary, build_segment_outputs, add_bands
from modeling import fit_models, score_portfolio, save_bundle

if not FREQ_PATH.exists() or not SEV_PATH.exists():
    download_data()

freq, sev = load_data(FREQ_PATH, SEV_PATH)
print("Frequency shape:", freq.shape)
print("Severity shape:", sev.shape)
freq.head()


## 2. Data quality and portfolio snapshot

The frequency and severity tables are linked through `IDpol`. The severity table contains fewer claim rows than the total `ClaimNb` count in the frequency table, so this notebook reports both:

- **Observed matched loss cost** = matched claim amount / exposure
- **Indicated pure premium** = observed annual frequency × average severity

The second measure follows the standard frequency-severity pricing framework.


In [ ]:
summary = portfolio_summary(freq, sev)
summary.T


### Verified full-dataset benchmark

The project package was verified against the full source data. Expected headline figures are approximately:

- 678,013 policies
- 358,499 policy-years of exposure
- 36,102 frequency claims
- 10.07 claims per 100 policy-years
- average severity about 2,265.51
- indicated pure premium about 228.14

Small differences can occur if you intentionally change cleaning rules.


## 3. Rating-factor segmentation

In [ ]:
segments = build_segment_outputs(freq, sev)

for name, table in segments.items():
    print("\n", name.upper())
    display(table.round(3))


In [ ]:
age = segments["driver_age"].sort_values("indicated_pure_premium")
plt.figure(figsize=(8, 4.5))
plt.bar(age["segment"].astype(str), age["indicated_pure_premium"])
plt.title("Indicated Pure Premium by Driver Age Band")
plt.xlabel("Driver age band")
plt.ylabel("Pure premium (dataset currency units)")
plt.tight_layout()
plt.show()


In [ ]:
bm = segments["bonus_malus"].sort_values("indicated_pure_premium")
plt.figure(figsize=(8, 4.5))
plt.bar(bm["segment"].astype(str), bm["indicated_pure_premium"])
plt.title("Indicated Pure Premium by Bonus-Malus Band")
plt.xlabel("Bonus-Malus band")
plt.ylabel("Pure premium (dataset currency units)")
plt.tight_layout()
plt.show()


## 4. Frequency model — Poisson GLM

Claim counts are modelled using a Poisson GLM with a log link. `log(Exposure)` is included as an offset so policies observed for different lengths of time are comparable on an annualised basis.

The model intentionally uses interpretable rating-factor bands rather than an opaque black-box model.


In [ ]:
bundle = fit_models(freq, sev, random_state=42)
bundle.metrics


## 5. Severity model — Gamma GLM

Positive claim amounts are continuous and strongly right-skewed. A Gamma GLM with log link is an interpretable actuarial starting point for severity.


In [ ]:
print(bundle.severity.summary())


## 6. Pure-premium predictions and risk tiers

In [ ]:
scored = score_portfolio(bundle, freq)
scored[["IDpol", "PredictedFrequency", "PredictedSeverity", "PurePremium", "RiskTier"]].head()


In [ ]:
tier_summary = scored.groupby("RiskTier", observed=True).agg(
    policies=("IDpol", "size"),
    avg_frequency=("PredictedFrequency", "mean"),
    avg_severity=("PredictedSeverity", "mean"),
    avg_pure_premium=("PurePremium", "mean"),
).reset_index()
tier_summary


## 7. Business interpretation

Use the descriptive one-factor tables as diagnostics, not as final rate factors. The GLM stage is important because driver age, Bonus-Malus, region, area and vehicle characteristics can be correlated.

Key questions for pricing governance:

1. Are the strongest relativities stable across train/test samples?
2. Is there evidence of overdispersion that justifies a Negative Binomial or quasi-Poisson alternative?
3. Are rating factors explainable and operationally acceptable?
4. Are any segments too small to support a stable rate?
5. How should expenses, commission, reinsurance, capital and profit be loaded onto pure premium?


## 8. Export model artifacts

The raw dataset is not committed to GitHub. Generated models are also ignored by default, but you can save them locally for interview demonstrations.


In [ ]:
save_bundle(bundle, MODEL_DIR)
print("Saved model artifacts to", MODEL_DIR)


## Limitations

- The data is historical French motor TPL and is not a current UK pricing portfolio.
- One-factor portfolio analysis can be confounded.
- The severity table has fewer claim rows than the total claim count in the frequency data; the project reports this transparently.
- A production pricing model requires stronger validation, governance, fairness review, pricing controls and commercial assumptions.
